In [17]:
import pandas as pd
import numpy as np
import fastparquet
import openpyxl

In [18]:
####################################################################################################################
# Carrega Sinan
sinan = pd.read_parquet('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Quadro1/Dados-processados/2_df_recodificado.parquet', 
                        filters=[('ANO', 'in', list(range(2019, 2025)))] # Com filtro interno
)

In [19]:
sinan['ANO'].unique()

array([2020, 2019, 2024, 2021, 2022, 2023], dtype=int32)

In [20]:
# Ajusta variaveis do SINAN

# Padroniza codigo municipio do Sinan como numero inteiro
sinan['ID_MN_RESI'] = sinan['ID_MN_RESI'].astype(int)

# Sinan para construir variavel MG
sinan_mg = sinan

# Converter ampolas para número
for col in ["NU_AMPOL_8", "NU_AMPOL_9"]:
    sinan_mg[col] = pd.to_numeric(sinan_mg[col], errors="coerce").fillna(0)

In [21]:
####################################################################################################################
# Cria variavel Moderado/Grave --(MG2)--

criterio_mg2 = (
    (sinan_mg["NU_AMPOL_8"] >= 2) & (sinan_mg["TRA_CLASSI"].isin(["Moderado", "Grave"]))|  # SAA, confirmar correspondência da coluna
    (sinan_mg["NU_AMPOL_9"] >= 2) & (sinan_mg["TRA_CLASSI"].isin(["Moderado", "Grave"])) |  # SAEsc
    (sinan_mg["EVOLUCAO"] == "Obito por ap") |
    (sinan_mg["CLI_VAGAIS"] == "Sim") |
    # (sinan_mg["TRA_CLASSI"].isin(["Moderado", "Grave"])) |
    (sinan_mg["MCLI_SIST"] == "Sim") |
    (sinan_mg["COM_SISTEM"] == "Sim")
)

sinan_mg["MG"] = criterio_mg2.astype(int)

In [22]:
# CRIAR COLUNAS DE TOTAIS

# TOTAIS GERAIS POR IDADE -------------------------------------------------------------------------------------
# Total de casos (sem distincao por faixa etaria)
total_casos = sinan_mg['ID_MN_RESI'].value_counts().reset_index(name='TOTAL_CASOS')

# Total ate 10 anos
total_10a = sinan_mg.loc[sinan_mg['IDADE_ANOS'] <= 10, 'ID_MN_RESI'].value_counts().reset_index(name='TOTAL_10A')

# Total ate 11 a 59 anos
total_11a59 = sinan_mg.loc[sinan_mg['IDADE_ANOS'].isin(range(11,59)), 'ID_MN_RESI'].value_counts().reset_index(name='TOTAL_11A59')

# Total 60+
total_60a = sinan_mg.loc[sinan_mg['IDADE_ANOS'] >59, 'ID_MN_RESI'].value_counts().reset_index(name='TOTAL_60')
# -------------------------------------------------------------------------------------------------------------

# TOTAIS DE MODERADOS/GRAVES POR IDADE ------------------------------------------------------------------------
# MG (total geral por municipio)
total_mg = sinan_mg.pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

total_mg.columns = ['ID_MN_RESI','LEVE','MG'] # Renomeia as colunas

# MG original (sem componente qualificado)
mg_original = sinan[sinan['TRA_CLASSI'].isin(['Moderado','Grave'])]
mg_original = mg_original.groupby('ID_MN_RESI').size().reset_index(name='MG_ORIGINAL').copy()

# MG ate 10
mg_10 = sinan_mg[sinan_mg['IDADE_ANOS']<=10].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_10.columns = ['ID_MN_RESI','LEVE_10','MG_10'] # Renomeia as colunas


# MG ate 12
mg_12 = sinan_mg[sinan_mg['IDADE_ANOS']<=12].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_12.columns = ['ID_MN_RESI','LEVE_12','MG_12'] # Renomeia as colunas

# MG 11 a 59
mg_11a59 = sinan_mg[sinan_mg['IDADE_ANOS'].isin(range(11,59))].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_11a59.columns = ['ID_MN_RESI','LEVE_11A59','MG_11A59'] # Renomeia as colunas


# MG 60+
mg_60 = sinan_mg[sinan_mg['IDADE_ANOS']>59].pivot_table(
    index='ID_MN_RESI', # linhas
    #values='ID_MN_RESI', # valores que serao contados
    columns='MG', # colunas
    fill_value=0, # valores ausentes serão preenchidos com zero
    aggfunc='size' # função de contagem
).reset_index()

mg_60.columns = ['ID_MN_RESI','LEVE_60','MG_60'] # Renomeia as colunas

# AMPOLAS
sinan_mg['AMPOLAS'] = sinan_mg['NU_AMPOL_8'] + sinan_mg['NU_AMPOL_9']
ampolas = sinan.groupby('ID_MN_RESI')['AMPOLAS'].sum().reset_index(name='TOTAL_AMPOLAS')


## COLUNAS
colunas = [total_casos,total_10a, total_11a59, total_60a, 
           total_mg, mg_original, mg_10, mg_11a59, mg_60, ampolas]



## Carregar codigos IBGE

- Objetivo: Juntar com variaveis do SINAN, para depois juntar à base02.

In [23]:
ibge = pd.read_excel('/home/usuario/Documentos/Lucas/Projetos estudos/Escorpiao_pesa/Quadro2/populacao2016-2023/pop_todos.xlsx')
# ibge = ibge.iloc[:,0:5].copy()

In [24]:
# Junta todas as colunas 

for df in colunas:
   ibge = ibge.merge(
        right=df,
        how='left',
        left_on='IBGE',
        right_on='ID_MN_RESI'
        ).drop(columns=['ID_MN_RESI'], errors='ignore')

In [25]:
ibge

,IBGE,POP10,POP11A59,POP60,POP_GERAL,MUNI,TOTAL_CASOS,TOTAL_10A,TOTAL_11A59,TOTAL_60,LEVE,MG,MG_ORIGINAL,LEVE_10,MG_10,LEVE_11A59,MG_11A59,LEVE_60,MG_60,TOTAL_AMPOLAS
0,350010,4246.000,23904.125,7418.500,35568.625,ADAMANTINA,260.0,43.0,136.0,76.0,221.0,39.0,39.0,16.0,27.0,128.0,8.0,73.0,3.0,185.0
1,350020,583.125,2801.000,825.250,4209.375,ADOLFO,93.0,8.0,53.0,30.0,91.0,2.0,5.0,6.0,2.0,53.0,0.0,30.0,0.0,6.0
2,350030,5154.500,23060.000,5072.250,33286.750,AGUAI,347.0,43.0,243.0,57.0,339.0,8.0,29.0,39.0,4.0,242.0,1.0,54.0,3.0,14.0
3,350040,950.000,4819.375,1857.375,7626.750,AGUAS DA PRATA,44.0,1.0,33.0,10.0,44.0,0.0,2.0,1.0,0.0,33.0,0.0,10.0,0.0,0.0
4,350050,2255.250,12329.750,3653.875,18238.875,AGUAS DE LINDOIA,205.0,7.0,151.0,42.0,198.0,7.0,4.0,5.0,2.0,146.0,5.0,42.0,0.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
640,355700,19778.500,90592.250,16471.250,126842.000,VOTORANTIM,254.0,41.0,170.0,40.0,223.0,31.0,20.0,28.0,13.0,156.0,14.0,36.0,4.0,27.0
641,355710,12604.250,66412.750,17228.500,96245.500,VOTUPORANGA,3425.0,286.0,2275.0,826.0,3331.0,94.0,81.0,220.0,66.0,2253.0,22.0,820.0,6.0,387.0
642,355715,390.750,1758.875,513.125,2662.750,ZACARIAS,73.0,4.0,55.0,13.0,71.0,2.0,2.0,4.0,0.0,54.0,1.0,12.0,1.0,3.0
643,355720,1933.500,8473.000,2118.000,12524.500,CHAVANTES,73.0,10.0,46.0,17.0,70.0,3.0,2.0,7.0,3.0,46.0,0.0,17.0,0.0,5.0


In [26]:
ibge.to_excel('dados/ibge.xlsx')

In [27]:
sinan_mg.to_csv('dados/sinan_mg2.csv',sep=';')